In [6]:
# STEP 1: Install and import libraries
!pip install nltk

import nltk
import re
import random
from nltk.corpus import twitter_samples
from nltk.tag import pos_tag
from nltk.stem.wordnet import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk import classify, NaiveBayesClassifier

# Download all required resources
nltk.download('twitter_samples')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger_eng') # Changed to download 'averaged_perceptron_tagger_eng'

# STEP 2: Load dataset (NLTK has sample tweets)
positive_tweets = twitter_samples.strings('positive_tweets.json')
negative_tweets = twitter_samples.strings('negative_tweets.json')
neutral_tweets  = twitter_samples.strings('tweets.20150430-223406.json')

print("Sample positive tweet:", positive_tweets[0])
print("Sample negative tweet:", negative_tweets[0])
print("Sample neutral tweet:", neutral_tweets[0])

# STEP 3: Preprocessing function
stop_words = stopwords.words('english')

def clean_tokens(tokens):
    lemmatizer = WordNetLemmatizer()
    cleaned = []
    for token, tag in pos_tag(tokens):
        token = re.sub(r'[^a-zA-Z]', '', token)  # remove non-letters
        if token and token.lower() not in stop_words:
            pos = 'n'
            if tag.startswith('V'):
                pos = 'v'
            elif tag.startswith('R'):
                pos = 'r'
            cleaned.append(lemmatizer.lemmatize(token.lower(), pos))
    return cleaned

def tokenize_tweet(tweet):
    tokens = nltk.word_tokenize(tweet)
    return clean_tokens(tokens)

# STEP 4: Feature extraction
def get_features(tokens):
    return {token: True for token in tokens}

positive_tokens = [tokenize_tweet(t) for t in positive_tweets]
negative_tokens = [tokenize_tweet(t) for t in negative_tweets]
neutral_tokens  = [tokenize_tweet(t) for t in neutral_tweets]

positive_dataset = [(get_features(tokens), "Positive") for tokens in positive_tokens]
negative_dataset = [(get_features(tokens), "Negative") for tokens in negative_tokens]
neutral_dataset  = [(get_features(tokens), "Neutral") for tokens in neutral_tokens]

dataset = positive_dataset + negative_dataset + neutral_dataset
random.shuffle(dataset)

# STEP 5: Train classifier
train_data = dataset[:7000]
test_data  = dataset[7000:]

classifier = NaiveBayesClassifier.train(train_data)

print("Accuracy:", classify.accuracy(classifier, test_data))
classifier.show_most_informative_features(10)

# STEP 6: Test on custom tweets
def predict_sentiment(tweet):
    tokens = tokenize_tweet(tweet)
    features = get_features(tokens)
    return classifier.classify(features)

sample_tweets = [
    "I love this new phone, it's amazing!",
    "This is the worst service ever.",
    "I am just sitting here, nothing much happening."
]

for tw in sample_tweets:
    print(f"Tweet: {tw} --> Sentiment: {predict_sentiment(tw)}")

[nltk_data] Downloading package twitter_samples to /root/nltk_data...
[nltk_data]   Package twitter_samples is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


Sample positive tweet: #FollowFriday @France_Inte @PKuchly57 @Milipol_Paris for being top engaged members in my community this week :)
Sample negative tweet: hopeless for tmr :(
Sample neutral tweet: RT @KirkKus: Indirect cost of the UK being in the EU is estimated to be costing Britain £170 billion per year! #BetterOffOut #UKIP
Accuracy: 0.8479130434782609
Most Informative Features
                      rt = True           Neutra : Positi =    537.9 : 1.0
                    tory = True           Neutra : Positi =    233.2 : 1.0
                  labour = True           Neutra : Positi =    113.1 : 1.0
                      hi = True           Positi : Neutra =     68.3 : 1.0
                  rather = True           Neutra : Positi =     60.4 : 1.0
                     wan = True           Negati : Neutra =     51.6 : 1.0
                  arrive = True           Positi : Neutra =     49.9 : 1.0
                 weekend = True           Positi : Neutra =     42.2 : 1.0
              